# The Probability of Passing a coin experiment

In [0]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

## FRECUENTIST WAY

In [0]:
def coin_experiment_demo (true_p = 0.7, n_samples = 10):
    """
    true_p : hidden bias of the coin  (unknown to students)
    n_samples: number of times a student flips the coin to learn
    """
    #1. THE EXPERIMENT: Collect realizations
    # 1 is head, 0 is tail

    # random.binomial genarates n_samples with 1 trial and p_true probability of success
    # tosses is a vector of 1 and 0
    tosses = np.random.binomial(1, true_p, n_samples)
    
    #Count the number of successes
    heads_observed = np.sum(tosses)
    
    #2. THE STIMATION: Frecuentist MLE (Maximum Likelihood Estimator)
    # Looking for the parameter p that maximizes the likelihood of the data
    # p is the probability of getting a head
    # tosses is the data
    # p_hat is the maximum likelihood estimator of p

    p_hat = heads_observed / n_samples
    

    #3. THE MODEL: Calculate P(pass) = p^5, get heads more than 4 times
    #Calculatin probability of passing the experiment
    true_pass_prob = true_p**5
    #Calculating probability of passing the experiment based on the estimator
    estimated_pass_prob = p_hat**5


    # --Visualization--
    # fig is the figure object
    # ax is the axis object
    fig, ax = plt.subplots(figsize= (10,5))

    #Plotting the function P(pass) = p^5
    p_range = np.linspace(0,1,100)
    pass_probs = p_range ** 5
    ax.plot(p_range, pass_probs, label = "P(pass)=p^5")

    #Mark truth vs the Estimate
    ax.scatter([true_p], [true_pass_prob], label = f"True logic (p = {true_p})", zorder =5)
    ax.scatter([p_hat], [estimated_pass_prob], color = 'red', marker = 'x', label = f"Estimate (p = {p_hat})", zorder =5)

    ax.set_title(f"Coin experiment with {n_samples} tosses")
    ax.set_xlabel("Coin Bias (p)")
    ax.set_ylabel("P(pass)")
    ax.legend()

    print(f"Realizations: {tosses}")
    print(f"Heads Observed: {heads_observed} / {n_samples}")
    print(f"True probs of passing: {true_pass_prob}")
    print(f"Estimated probs of passing: {estimated_pass_prob}")
    

    plt.show()





    

In [0]:
coin_experiment_demo(true_p=0.5, n_samples=1)

In [0]:
coin_experiment_demo(true_p=0.5, n_samples=10)

In [0]:
coin_experiment_demo(true_p= 0.7, n_samples=10)


In [0]:
coin_experiment_demo(true_p=0.25, n_samples=10)

## BAYESIAN WAY

# Bayesian Estimation with Beta Prior

## 1. Coin Toss Model

We model each coin toss as a Bernoulli random variable:

$$
x_i \sim \text{Bernoulli}(p)
$$

where:

| Symbol | Meaning |
|---|---|
| $$x_i = 1$$ | Head |
| $$x_i = 0$$ | Tail |
| $$p$$ | Probability of getting head |

If we observe $$n$$ tosses, the total number of heads is:

$$
h = \sum_{i=1}^{n} x_i
$$

Therefore:

| Symbol | Meaning |
|---|---|
| $$h$$ | Number of heads |
| $$n$$ | Total number of tosses |
| $$n - h$$ | Number of tails |

---

## 2. Beta Prior

In Bayesian estimation, we assume a prior belief about $$p$$:

$$
p \sim \text{Beta}(\alpha, \beta)
$$

where:

| Parameter | Meaning |
|---|---|
| $$\alpha$$ | Prior evidence for heads |
| $$\beta$$ | Prior evidence for tails |

---

## 3. Posterior Distribution

After observing the data, the prior is updated into a posterior distribution:

$$
p \mid \text{data} \sim \text{Beta}(\alpha + h,\ \beta + n - h)
$$

This means:

$$
\text{Posterior} = \text{Prior} + \text{Observed Data}
$$

---

## 4. MAP Estimate

The MAP estimate is the most likely value of $$p$$ after seeing the data:

$$
p_{\text{MAP}} =
\frac{\alpha + h - 1}
{\alpha + \beta + n - 2}
$$

---

## 5. Comparison with Frequentist MLE

The frequentist MLE only uses the observed data:

$$
p_{\text{MLE}} = \frac{h}{n}
$$

The Bayesian MAP uses both the observed data and the prior belief:

$$
p_{\text{MAP}} =
\frac{\alpha + h - 1}
{\alpha + \beta + n - 2}
$$

---

## Key Idea

| Method | Uses |
|---|---|
| MLE | Only observed data |
| MAP | Observed data + prior belief |

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binom, beta

def bayesian_coin_demo(true_p = 0.6, n_samples = 10, alpha_prior = 2, beta_prior = 2):
    """
    alpha_prior, beta_prior: hyperparameters of the prior beta distribution
    (2,2) is a weak 'fair coin' prior. (50,50) is strong prior
    """
    #1. THE EXPERIMENT: Realizations
    tosses = np.random.binomial(1, true_p, n_samples)
    heads = np.sum(tosses)
    tails = n_samples - heads

    #2. ESTIMATION
    #Frecuentist MLE
    p_mle = heads / n_samples

    #Bayesian (MAP estimation with Beta prior)
    #The posterior of a bernuilli  with Beta prior is Beta(alpha + Heads, beta + tails)
    #Prior Map
    p_map_prior = (alpha_prior - 1) / (alpha_prior + beta_prior - 2)

    #Posterior Map
    p_map_posterior = (heads + alpha_prior - 1) / (n_samples + alpha_prior + beta_prior - 2)

    #Print realizations and estimates
    print('Realizations:', tosses)
    print('Heads', heads)
    print('True Bias', true_p)
    print('Frecuentist MLE', p_mle)
    print('Bayesian Prior', p_map_prior)
    print('Bayesian Posterior', p_map_posterior)
    
    # PROBABILITY LANDSCAPE
    p_axis = np.linspace(0,1,100)

    # THEORETHICAL CURVE P(Pass)= p^5
    plt.title("Bayesian Coin Demo: Estimating p and P(pass)")
    plt.plot(p_axis, p_axis**5, linestyle='--', label = r"Model: $P(pass)=p^5$")

    #THE BAYESIAN PRIOR AND POSTERIOR DISTRIBUTIONS
    prior_pdf = beta.pdf(p_axis, alpha_prior, beta_prior)
    plt.fill_between(p_axis, 0, prior_pdf / prior_pdf.max(), alpha=0.2, label = "Bayesian Prior")

    posterior_pdf = beta.pdf(p_axis, alpha_prior + heads, beta_prior + tails)
    plt.fill_between(p_axis, 0, posterior_pdf / posterior_pdf.max(), alpha = 0.2, label= "Bayesian Posterior")


    plt.scatter([true_p], [true_p**5], color='black', label=f"True prob: {true_p}")
    plt.scatter([p_mle], [p_mle**5], color='red', marker='x', label=f"Frecuentist MLE: {p_mle}")
    plt.scatter([p_map_prior], [p_map_prior**5], color='blue', label=f"Bayesian Prior: {p_map_prior}")
    plt.scatter([p_map_posterior], [p_map_posterior**5], color='orange', label=f"Bayesian Posterior: {p_map_posterior}")

    plt.title("Bayesian Coin Demo: Estimating p and P(pass)")
    plt.xlabel("Coin Bias (p)")
    plt.ylabel("P(pass)")

    plt.grid(True)


    plt.legend()
    plt.show()



### Weak Samples and Prior alpha and Beta

In [0]:
bayesian_coin_demo(true_p = 0.7, n_samples = 10, alpha_prior = 2, beta_prior = 2)

### Bigger number of Samples and Weak beta and alpha prior

In [0]:
bayesian_coin_demo(true_p = 0.7, n_samples = 100, alpha_prior = 2, beta_prior = 2)

### Weak number of Samples and  Strong beta prior parameters

In [0]:
bayesian_coin_demo(true_p = 0.7, n_samples = 10, alpha_prior = 50, beta_prior = 50)

### Bigger Samples and Better Parameters of Beta prior Distribution

In [0]:
bayesian_coin_demo(true_p = 0.7, n_samples = 100, alpha_prior = 50, beta_prior = 50)